In [1]:
from transformers import CLIPModel, CLIPProcessor
import torch
from PIL import Image
import os
import torch.nn.functional as F
import numpy as np
import pandas as pd

In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM

## IMPORT & EXPLORE

#### GEMMA SETUP

In [3]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

gemma_1b = Gemma3ForCausalLM.from_pretrained(
    "google/gemma-3-1b-it", quantization_config=quantization_config
).eval()

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

#### CLIP SETUP

In [4]:
# Load pretrained CLIP
clip_model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_id)
clip_processor = CLIPProcessor.from_pretrained(clip_model_id, use_fast = True)

In [5]:
clip_model = clip_model.to('cuda')

In [6]:
clip_model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

#### IMAGE DIR SETUP

In [7]:
class ImageFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        # recursively collect all image files in all subfolders
        self.image_files = []
        for root, _, files in os.walk(root_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    self.image_files.append(os.path.join(root, f))
        
    
    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        
        image_path = self.image_files[index]
        image = Image.open(image_path).convert('RGB')
        
        clip_processed_image = clip_processor( images = image, return_tensors = 'pt')

        return clip_processed_image, image_path


In [8]:
image_dataset = ImageFolderDataset(r"C:\Users\User\Downloads\data\images")

In [9]:
len(image_dataset)

13731

In [10]:
image_dataset[42][0]

{'pixel_values': tensor([[[[1.6530, 1.6530, 1.6530,  ..., 1.6092, 1.5946, 1.5800],
          [1.6530, 1.6530, 1.6530,  ..., 1.6092, 1.5946, 1.5800],
          [1.6530, 1.6530, 1.6530,  ..., 1.6092, 1.5946, 1.5800],
          ...,
          [1.6238, 1.6092, 1.5946,  ..., 1.5362, 1.5362, 1.5362],
          [1.6238, 1.6092, 1.5946,  ..., 1.5362, 1.5362, 1.5362],
          [1.6238, 1.6092, 1.5946,  ..., 1.5362, 1.5362, 1.5362]],

         [[1.8047, 1.8047, 1.8047,  ..., 1.7447, 1.7297, 1.7147],
          [1.8047, 1.8047, 1.8047,  ..., 1.7447, 1.7297, 1.7147],
          [1.8047, 1.8047, 1.8047,  ..., 1.7447, 1.7297, 1.7147],
          ...,
          [1.7597, 1.7447, 1.7297,  ..., 1.6847, 1.6847, 1.6847],
          [1.7597, 1.7447, 1.7297,  ..., 1.6847, 1.6847, 1.6847],
          [1.7597, 1.7447, 1.7297,  ..., 1.6847, 1.6847, 1.6847]],

         [[1.8473, 1.8473, 1.8473,  ..., 1.8046, 1.7904, 1.7762],
          [1.8473, 1.8473, 1.8473,  ..., 1.8046, 1.7904, 1.7762],
          [1.8473, 1.8473

In [14]:
images_dataloader = torch.utils.data.DataLoader( dataset = image_dataset, 
                                                 batch_size = 64 )

## COMPUTING EMBEDDINGS

In [15]:
image_embeddings_list = []
image_paths_list = []

clip_model.eval()

with torch.no_grad():
    for batch_num, (batch_images, batch_image_paths) in enumerate(images_dataloader):
        # Move to GPU
        batch_images = {k: v.to("cuda") for k, v in batch_images.items()}
        
        # Fix extra dimension if present
        pixel_values = batch_images["pixel_values"]
        if pixel_values.ndim == 5:   # [batch, 1, 3, H, W]
            pixel_values = pixel_values.squeeze(1)  # -> [batch, 3, H, W]
            # Update the batch_images dictionary with the corrected tensor
            batch_images["pixel_values"] = pixel_values
        
        # Get embeddings from CLIP
        image_embeddings = clip_model.get_image_features(**batch_images)
        
        # Normalize (important for cosine sim later)
        image_embeddings = image_embeddings / image_embeddings.norm(p=2, dim=-1, keepdim=True)
        
        # Store Paths and Image embeddings 
        image_embeddings_list.append(image_embeddings)
        image_paths_list.extend(batch_image_paths)
        
        print(f'Processed {batch_num + 1} / {len(images_dataloader)}')

Processed 1 / 215
Processed 2 / 215
Processed 3 / 215
Processed 4 / 215
Processed 5 / 215
Processed 6 / 215
Processed 7 / 215
Processed 8 / 215
Processed 9 / 215
Processed 10 / 215
Processed 11 / 215
Processed 12 / 215
Processed 13 / 215
Processed 14 / 215
Processed 15 / 215
Processed 16 / 215
Processed 17 / 215
Processed 18 / 215
Processed 19 / 215
Processed 20 / 215
Processed 21 / 215
Processed 22 / 215
Processed 23 / 215
Processed 24 / 215
Processed 25 / 215
Processed 26 / 215
Processed 27 / 215
Processed 28 / 215
Processed 29 / 215
Processed 30 / 215
Processed 31 / 215
Processed 32 / 215
Processed 33 / 215
Processed 34 / 215
Processed 35 / 215
Processed 36 / 215
Processed 37 / 215
Processed 38 / 215
Processed 39 / 215
Processed 40 / 215
Processed 41 / 215
Processed 42 / 215
Processed 43 / 215
Processed 44 / 215
Processed 45 / 215
Processed 46 / 215
Processed 47 / 215
Processed 48 / 215
Processed 49 / 215
Processed 50 / 215
Processed 51 / 215
Processed 52 / 215
Processed 53 / 215
Pr

In [16]:
image_embeddings_list

[tensor([[-1.3460e-02, -2.8147e-02,  4.9930e-03,  ...,  8.0066e-02,
          -4.2377e-03,  5.6023e-02],
         [-2.4412e-02, -1.7838e-02, -1.9038e-03,  ...,  6.7288e-02,
          -2.9393e-02,  3.9701e-02],
         [-3.3199e-02, -1.0842e-02,  2.1525e-02,  ...,  5.0295e-02,
          -1.6612e-02,  3.4842e-02],
         ...,
         [ 2.1666e-03, -5.4886e-03,  1.4450e-02,  ...,  2.4119e-03,
          -9.5442e-03,  5.4990e-02],
         [-6.7434e-03,  5.0550e-03,  3.8905e-02,  ...,  1.6957e-02,
           1.1302e-05,  3.4470e-02],
         [ 2.8318e-03,  2.8710e-02,  2.1019e-02,  ...,  7.5401e-02,
          -3.4803e-02,  2.5562e-02]], device='cuda:0'),
 tensor([[-0.0032,  0.0163,  0.0412,  ...,  0.0549, -0.0118,  0.0278],
         [ 0.0152,  0.0014,  0.0251,  ...,  0.0072,  0.0107,  0.0243],
         [ 0.0030, -0.0140,  0.0391,  ...,  0.0383,  0.0007,  0.0451],
         ...,
         [-0.0307,  0.0052,  0.0172,  ...,  0.0706, -0.0251,  0.0247],
         [-0.0271, -0.0079,  0.0329,  .

In [17]:
all_image_embeddings = torch.cat(image_embeddings_list, dim=0)

In [18]:
all_image_embeddings.shape

torch.Size([13731, 512])

In [19]:
torch.save( {'image_embeddings' : all_image_embeddings.cpu(),
             'image_paths' : image_paths_list},
             
             r'C:\Users\User\Downloads\clip_image_embeddings.pt'  )

## SEARCH REQUEST

#### LOAD DATA

In [20]:
clip_image_data = torch.load(r"C:\Users\User\Downloads\clip_image_embeddings.pt")

In [21]:
clip_image_embeddings = clip_image_data['image_embeddings']
image_paths = clip_image_data['image_paths']

In [22]:
clip_image_embeddings.shape[0] == len(image_paths)

True

#### SEND REQUEST TO GEMMA

In [ ]:
messages = [
        {
            "role": "system",
            "content": (
                "You are a strict fashion assistant specializing in **men's and women's clothing**. "
                "You must ALWAYS respect the user's requested gender and colors/themes. "
                "Your only output must be **exactly one valid Python list**, with no text, no formatting, no explanations. "
                ""
                "STRICT RULES: "
                "1. The list must contain **minimum 3 and maximum 6 items**. If you cannot meet this, output nothing. "
                "2. Each item must be a **single clothing or accessory description**. "
                "3. Each description must include **specific details** (colors, fabrics, styles). "
                "4. Each item must belong to **different categories** (bottoms, tops, outer layers, shoes, optional accessories). "
                "   - Only ONE item per category. "
                "5. If the user specifies a gender, you MUST return clothing appropriate for that gender. "
                "6. You must include the gender in every item description, in the format: 'men ...' or 'women ...'. "
                "7. If the user specifies a color or theme, ALL items must strictly follow it. NO EXCEPTIONS. "
                "   - 'total black' means EVERY item must be black. "
                "   - 'all white' means EVERY item must be white. "
                "   - 'monochrome blue' means EVERY item must be blue. "
                "8. You must NEVER output fewer than 3 or more than 6 items. "
                "9. Use only plain strings inside the Python list. "
                "10. NEVER repeat categories - one bottom, one top, one outer layer, one shoe maximum. "
                ""
                "ALLOWED CATEGORIES: "
                "- BOTTOMS: pants, trousers, jeans, chinos, shorts, joggers, skirts, dresses "
                "- TOPS: t-shirts, polo shirts, blouses, crop tops, shirts "
                "- OUTER LAYERS: hoodies, sweatshirts, zip-hoodies, cardigans, jackets, coats "
                "- SHOES: sneakers, loafers, boots, dress shoes, canvas shoes, running shoes, casual shoes, heels, sandals "
                "- ACCESSORIES (optional, max 2): hats, caps, glasses, watches, belts, bags, jewelry, scarves "
                ""
                "SEASONAL RULES: "
                "- WARM weather: lighter fabrics, breathable pieces, skirts, dresses, crop tops, sandals. "
                "- COLD weather: thick fabrics, warm outer layers, boots, scarves, cozy accessories. "
                ""
                "CRITICAL COLOR ENFORCEMENT: "
                "When user specifies a color theme, you must check each item contains that exact color. "
                "Examples of CORRECT responses: "
                "- 'total black for men' → ['men black cotton t-shirt', 'men black denim jeans', 'men black leather boots', 'men black bomber jacket'] "
                "- 'all white for women' → ['women white cotton blouse', 'women white linen pants', 'women white canvas sneakers'] "
                ""
                "Examples of WRONG responses (DO NOT DO): "
                "- 'total black for men' → ['men black shirt', 'men beige chinos', 'men white sneakers'] (violates color rule) "
                "- Any response with repeated categories like jeans AND chinos (both bottoms) "
                ""
                "OUTPUT FORMAT: "
                "Return ONLY a valid Python list (e.g., ['men black linen shirt', 'men black chinos', 'men black sneakers']). "
                "NO explanations, NO text before or after, NO code blocks, NO markdown formatting. "
            )
        },
        {
            "role": "user",
            "content": ("total white")
        }
    ]


In [30]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to('cuda')

with torch.inference_mode():
    outputs = gemma_1b.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)

In [31]:
outputs

["<bos><start_of_turn>user\nYou are a strict fashion assistant specializing in **men's and women's clothing**. You must ALWAYS respect the user’s requested gender and colors/themes. Your only output must be **exactly one valid Python list**, with no text, no formatting, no explanations. STRICT RULES: 1. The list must contain **minimum 3 and maximum 6 items**. If you cannot meet this, output nothing. 2. Each item must be a **single clothing or accessory description**. 3. Each description must include **specific details** (colors, fabrics, styles). 4. Each item must belong to **different categories** (bottoms, tops, outer layers, shoes, optional accessories).    - Only ONE item per category. 5. If the user specifies a gender, you MUST return clothing appropriate for that gender. 6. You must include the gender in every item description, in the format: 'men's ...' or 'women's ...'. 7. If the user specifies a color or theme, ALL items must strictly follow it. 8. You must NEVER output fewer 

In [17]:
sum(p.numel() for p in gemma_1b.parameters())

999885952